# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vikraamkumar-ds/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

**Lane: Refresh / Decline scoring.** Unit of analysis = one `content_hash_id` for one client. Label and features both come from the same 60-day panel used in Notebook 3, split at the midpoint:

- **Feature window (days 60→31 before the panel's max date):** `imp_prev30`, `clk_prev30`, `pos_prev30`, engineered `ctr_prev30 = clk_prev30 / imp_prev30`, plus query-mix signals from `fact_content_query_90d` (`visible_queries`, `rare_share`, `anon_share`, `top_query_share`).
- **Outcome window (days 30→0 before the panel's max date):** `imp_last30` — used ONLY to build the label, never as a feature.
- **Label:** `is_declining = 1` if `imp_last30 < 0.8 * imp_prev30`, else `0`.
- **Categorical handling:** no categorical columns in this feature set — everything is a count, a rate, or a share, so no encoding needed.
- **Missing/fill policy:** rows below `imp_prev30 >= 100` are dropped at the SQL stage (too little history to trust a rate off of); any remaining NaNs (e.g. `pos_prev30` when a page had zero impression-days in the window) are dropped rather than filled — filling a position with 0 or the mean would fabricate a signal that didn't exist.


In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, getpass
import duckdb
import pandas as pd

# token order: env var -> Colab Secret -> manual prompt (same pattern as notebook 03)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# --- prev-30 features only (the outcome window's imp_last30 shows up ONLY in the label below) ---
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)  AS visible_queries,
           ANY_VALUE(rare_impressions_share)        AS rare_share,
           ANY_VALUE(anonymized_impressions_share)  AS anon_share,
           MAX(impressions_90d)                     AS top_query_impressions,
           SUM(impressions_90d)                     AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()
qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

data = features.merge(qsignals, on='content_hash_id', how='left')

# --- the label: built from the OUTCOME window, never included in FEATURE_COLS ---
data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

FEATURE_COLS = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
                'visible_queries', 'rare_share', 'anon_share', 'top_query_share']

feature_vector = data.dropna(subset=FEATURE_COLS + ['is_declining']).copy()
print(f'{len(feature_vector):,} rows, {len(FEATURE_COLS)} features, '
      f'base rate = {feature_vector["is_declining"].mean():.3f}')
feature_vector[FEATURE_COLS + ['is_declining']].describe()


## 2. Feature notes (meaning, missing, categorical, available-when?)

| feature | meaning | missing handling | available before outcome window? |
|---|---|---|---|
| `imp_prev30` | GSC impressions, days 60→31 before panel end | rows with <100 dropped at SQL stage | Yes — entirely prior period |
| `clk_prev30` | GSC clicks, same window | same as above | Yes |
| `pos_prev30` | avg. search position, same window | NaN if zero impression-days in window → row dropped | Yes |
| `ctr_prev30` | engineered: `clk_prev30 / imp_prev30` | inherits parents' handling; undefined only if `imp_prev30` were 0, which the `>=100` filter prevents | Yes — derived only from prev30 columns |
| `visible_queries` | count of distinct queries a page ranks for | left-join NaN if content has no `fact_content_query_90d` row → dropped by `dropna` | See caveat below |
| `rare_share`, `anon_share` | share of impressions in the rare/anonymized long tail | same as above | See caveat below |
| `top_query_share` | concentration — how much of a page's impressions sit in its single top query | same as above | See caveat below |

**Caveat on the query-mix features:** `fact_content_query_90d` is a trailing 90-day window ending at the same panel date as `fact_daily`. Since 90 days > 30 days, this window **overlaps with the outcome (last-30) period**, not just the feature (prev-30) period. This is examined directly in Section 3.


In [ ]:
# Confirm the missingness story told above, numerically
print('Rows before dropna:', len(data))
print('Rows after dropna (final feature_vector):', len(feature_vector))
print()
print('NaN count per feature, before dropna:')
print(data[FEATURE_COLS].isna().sum())
print()
print('ctr_prev30 sanity check — min/max should sit in [0, ~1] since HAVING imp_prev30>=100 rules out div-by-zero:')
print(feature_vector['ctr_prev30'].describe()[['min', 'max']])


## 3. The leakage hunt

Three attacks against my own feature set:

**Test 1 — does the label-defining column sit inside the feature set?** `imp_last30` builds `is_declining` and must never appear in `FEATURE_COLS`. Checked with a hard assertion below.

**Test 2 — single-feature AUC sweep.** If any one feature alone predicts the label almost perfectly (AUC > 0.90), that is a leakage red flag, not a good feature — real signals are noisy and partial, not deterministic.

**Test 3 — the query-mix overlap caveat from Section 2, tested directly.** Since `fact_content_query_90d` covers 90 trailing days and the outcome window is the last 30 of that panel, the query-mix features are **not strictly pre-outcome** — they partially describe the same period the label is measuring. I could not find a strictly prior-only query-mix table in this release. **Decision:** I keep these features but explicitly downgrade the paper's claims about them to *directional / descriptive*, not predictive — and flag this as a named limitation rather than silently proceeding as if they were clean.


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

# Test 1: the outcome column must not be a feature
assert 'imp_last30' not in FEATURE_COLS, 'LEAKAGE: outcome column found inside FEATURE_COLS'
print('PASS — imp_last30 (used to build the label) is excluded from FEATURE_COLS')

# Test 2: single-feature AUC sweep
X = feature_vector[FEATURE_COLS]
y = feature_vector['is_declining']

print(f"\n{'feature':<18}{'single-feature AUC':>20}")
for col in FEATURE_COLS:
    auc = roc_auc_score(y, X[[col]].rank(pct=True).iloc[:, 0])
    flag = '  <-- investigate, looks too good' if auc > 0.90 else ''
    print(f'{col:<18}{auc:>20.3f}{flag}')

# Test 3: explicit window-overlap check on the query-mix table
panel_bounds = con.sql(f"SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {TABLES['fact_daily']}").df()
end_d = panel_bounds['max_d'].iloc[0]
outcome_start = end_d - pd.Timedelta(days=30)
query_window_start = end_d - pd.Timedelta(days=90)
print(f"\nPanel end date:        {end_d}")
print(f"Outcome window starts: {outcome_start}")
print(f"Query-mix (90d) window starts: {query_window_start}")
print('Overlap between query-mix window and outcome window:',
      query_window_start < outcome_start,
      '-- confirmed: query-mix features are NOT strictly pre-outcome, see caveat in Section 2/3.')


## 4. What I excluded and why


In [ ]:
excluded_fields = {
    'content_hash_id / client_hash_id': 'identifiers, not signal — keeping them as features risks the model '
        'memorizing specific pages/clients instead of learning a pattern that generalizes to new content.',
    'imp_last30 / clk_last30': 'these directly define the label — using them as features would be pure leakage '
        'by construction.',
    'report_date (raw)': 'a raw calendar date lets a model key off *when* rather than *what* — not a '
        'generalizable signal. The prev30/last30 windowing already encodes the relevant time structure.',
    'gsc_data_start / ga4_data_start (dim_clients)': 'proxies for a client\'s onboarding cohort/tenure, not '
        'content behavior — a model could use these as a shortcut to identify specific clients rather than '
        'learning real refresh signal.',
    'access_profile (dim_clients)': 'client-level metadata, not content-level — same generalization risk as '
        'client_hash_id above.',
}

for field, reason in excluded_fields.items():
    print(f'- {field}:\n    {reason}\n')


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.